In [1]:
import gymnasium as gym

import math 
import random 

import matplotlib.pyplot as plt
%matplotlib inline

from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [2]:
env = gym.make("CartPole-v1", render_mode = "human")

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
Transition = namedtuple("Transition", ("state", "action", "next_state", "reward"))

class ReplayMemory(object):
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)
    
    def __len__(self):
        return len(self.memory)

In [ ]:
# Test case: Sampling with small memory
small_memory = ReplayMemory(5)
small_memory.push(1, 2, 3, 4)
small_memory.push(5, 6, 7, 8)
small_memory.push(9, 10, 11, 12)

sample = small_memory.sample(2)
print(sample)


In [6]:
class DQN(nn.Module):

    def __init__(self, n_observations, n_actions):

        super(DQN, self).__init__()
        
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    def forward(self, x):

        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        
        return self.layer3(x)

In [ ]:
BATCH_SIZE = 128
GAMMA = 0.99

EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 1000

TAU = 0.005
LR = 1e-4

n_actions = env.action_space.n
state, info = env.reset()

n_observations = len(state)

policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)

target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10000)



In [ ]:
steps_done = 0

def select_action(state):
    global steps_done
    
    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(-1. * steps_done/ EPS_DECAY)

    steps_done += 1

    if sample > eps_threshold:
        with torch.no_grad():
            return policy_net(state).max(1).indices.view(1, 1)
        
    else:
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)

In [ ]:
episode_duration = []

def plot_duration(show_result = False):
    plt.figure(1)
    duration_t = torch.tensor(episode_duration, device=device, dtype=torch.float)

    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title("Training...")
        plt.xlabel("Episode")
        plt.ylabel("Duration")

        plt.plot(duration_t.numpy())

        if len(duration_t) >= 100:
            means = duration_t.unfold(0, 100, 1).mean(1).view(-1)
            means = torch.cat((torch.zeros(99), means))
            plt.plot(means.numpy())

        plt.pause(0.001)

In [ ]:
def optimize_model():
    if len(memory) < BATCH_SIZE:
        return
    transitions = memory.sample(BATCH_SIZE)  
    batch = Transition(*zip(*transitions))

    non_final_mask = torch.tensor(tuple(map(lambda x: x is not None, batch.next_state)), device=device, dtype=torch.bool)
    non_final_next_state = torch.cat([x for x in batch.next_state if x is not None])

    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    state_action_value = policy_net(state_batch).gather(1, action_batch)
    next_state_value = torch.zeros(BATCH_SIZE, device=device)
    

In [ ]:
num_episodes = 100

for i_episode in range(num_episodes):
    state, info = env.reset()

    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    for i in count():
        action = select_action(state)
        observation, reward, terminated, truncated, = env.step(action.item())

        reward = torch.tensor([reward], device=device)
        done = terminated or truncated

        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        # store transition state & move to next state
        memory.push(state, action, next_state, reward)
        state = next_state

        # perform one step optimzation (policy network)
        optimize_model()

        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()

        # soft update of the target network weights
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key] * TAU + target_net_state_dict[key] * (1 - TAU)
        
        target_net.load_state_dict(target_net_state_dict)

        if done:
            episode_duration.append(i + 1)
            plot_duration()
            break

print('complete')

plot_duration(show_result=True)
plt.ioff()
plt.show()